# Week 5 — Polysemantic Neurons & Sparse Autoencoders

> Training a from-scratch Sparse Autoencoder to decompose polysemantic activations into monosemantic features, then mapping their co-activation structure with a network graph.

---

## 1. Theory

### 1.1 The superposition hypothesis

Anthropic's superposition framework (Elhage et al., 2022) posits that LLMs represent **more features than dimensions**. With $d$-dimensional activations and $F \gg d$ semantically meaningful features, the model encodes a sparse code $z \in \mathbb{R}^F$ through a (near-)orthogonal dictionary $D \in \mathbb{R}^{d \times F}$:

$$x \;=\; D z + \eta, \qquad \|z\|_0 \ll F$$

Each neuron is **polysemantic** — its activation correlates with many different latent concepts because the dictionary atoms project onto multiple neurons simultaneously.

### 1.2 Sparse Autoencoder formulation

The SAE inverts this superposition by learning an overcomplete encoder/decoder:

$$z = \mathrm{ReLU}\bigl(W_\mathrm{enc}\,(x - b_\mathrm{dec}) + b_\mathrm{enc}\bigr), \qquad \hat{x} = W_\mathrm{dec}\,z + b_\mathrm{dec}$$

with $W_\mathrm{enc} \in \mathbb{R}^{F \times d}$ and $W_\mathrm{dec} \in \mathbb{R}^{d \times F}$, $F = m \cdot d$ for an expansion factor $m \in [4, 32]$. The loss is

$$\mathcal{L}_\mathrm{SAE}(x) = \underbrace{\|x - \hat{x}\|_2^2}_{\text{reconstruction}} + \lambda \underbrace{\|z\|_1}_{\text{L1 sparsity}}$$

Critical implementation details (from Bricken et al., 2023):

* **Decoder unit norm**: project $W_\mathrm{dec}[:, k]$ onto $\mathbb{S}^{d-1}$ after every step. Otherwise the optimizer can shrink $W_\mathrm{dec}$ and grow $z$ to trivially satisfy the L1.
* **Gradient projection**: remove the component of $\nabla W_\mathrm{dec}$ parallel to $W_\mathrm{dec}$ to keep the normalization constraint exact.
* **Dead feature resampling**: features that never activate for thousands of steps are reinitialized to randomly sampled (and renormalized) activation vectors.

### 1.3 Co-occurrence graph

Once features are learned, their **Jaccard co-activation** captures circuit structure:

$$J(k, l) \;=\; \frac{|\{i : z_{i,k} > 0\} \cap \{i : z_{i,l} > 0\}|}{|\{i : z_{i,k} > 0\} \cup \{i : z_{i,l} > 0\}|}$$

Features that frequently fire together are part of the same conceptual circuit; the graph layout (force-directed) reveals these clusters.

### 1.4 Integrated gradients (background)

For attribution at the feature level, Sundararajan et al. (2017) define the integrated gradient of feature $k$ at input $x$ relative to a baseline $x'$ as

$$\mathrm{IG}_k(x) \;=\; (x_k - x'_k) \int_0^1 \frac{\partial F(x' + \alpha(x - x'))}{\partial x_k}\, d\alpha$$

— a path integral satisfying completeness, sensitivity, and symmetry axioms. We do not implement IG here, but the framework feeds directly from SAE outputs.

References
----------
* Elhage, N. et al. (2022). *Toy Models of Superposition.* Anthropic.
* Bricken, T. et al. (2023). *Towards Monosemanticity: Decomposing Language Models with Dictionary Learning.* Anthropic.
* Sundararajan, M., Taly, A., Yan, Q. (2017). *Axiomatic Attribution for Deep Networks.* ICML.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

from src.utils import set_global_seed
from src.utils.synthetic import synth_polysemantic_activations
from src.mechanistic import (
    SAETrainingConfig,
    SparseAutoencoder,
    build_feature_cooccurrence_graph,
    top_activating_examples,
    train_sae,
)

set_global_seed(0)
print("TensorLens — Week 5 notebook loaded")


## 2. Synthesizing polysemantic activations

We plant $F = 256$ ground-truth features in a $d = 32$-dim activation space (expansion ratio 8). Each sample fires a sparse subset (density 4%) of those features through a random unit-norm dictionary, then we add small isotropic noise. The SAE's job is to recover the 256 features.

In [ ]:
N_SAMPLES = 8000
D_MODEL = 32
N_FEATURES_TRUE = 256
SPARSITY = 0.04

activations, true_codes = synth_polysemantic_activations(
    n_samples=N_SAMPLES,
    d_model=D_MODEL,
    n_features=N_FEATURES_TRUE,
    sparsity=SPARSITY,
    seed=42,
)
print(f"Activations shape: {tuple(activations.shape)}")
print(f"True codes density: {(true_codes > 0).float().mean().item():.4f}  (target ≈ {SPARSITY})")
print(f"Activation norm distribution: mean={activations.norm(dim=1).mean():.3f}, std={activations.norm(dim=1).std():.3f}")


## 3. Training the SAE

We use a dictionary of $F = 384$ atoms (slightly more than the true 256 — overcomplete) and train with L1 coefficient $\lambda = 10^{-2}$ over 8 epochs.

In [ ]:
N_FEATURES_SAE = 384

sae = SparseAutoencoder(d_model=D_MODEL, n_features=N_FEATURES_SAE)
config = SAETrainingConfig(
    l1_coefficient=1e-2,
    learning_rate=1e-3,
    batch_size=256,
    n_epochs=8,
    dead_feature_window=400,
    dead_feature_threshold=1e-5,
    resample_dead=True,
    seed=0,
)

result = train_sae(activations, sae, config)
print(f"Final loss:           {result.losses[-1]:.5f}")
print(f"Final recon loss:     {result.reconstruction_losses[-1]:.5f}")
print(f"Final sparsity loss:  {result.sparsity_losses[-1]:.3f}")
print(f"Dead features:        {result.n_dead_features} / {N_FEATURES_SAE}")


## 4. Training curves

The reconstruction loss should decrease rapidly; the sparsity loss should plateau around the target density × n_features.

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Reconstruction loss", "Sparsity loss (mean L1 of code)"))
fig.add_trace(go.Scatter(y=result.reconstruction_losses, mode="lines", name="recon"),
              row=1, col=1)
fig.add_trace(go.Scatter(y=result.sparsity_losses, mode="lines", name="sparsity",
                         line=dict(color="firebrick")),
              row=1, col=2)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_xaxes(title="step")
fig.update_layout(height=380, width=1100, title="SAE training curves",
                  showlegend=False)
fig.show()


## 5. Feature activation density distribution

A healthy SAE has a long tail of densities — most features fire rarely but specifically; dead features (density < threshold) are a failure mode.

In [ ]:
density = result.feature_activation_density.numpy()
density_sorted = np.sort(density)[::-1]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Density histogram (log scale)", "Sorted feature density (Zipf-style plot)"))
fig.add_trace(go.Histogram(x=np.log10(density.clip(min=1e-8)), nbinsx=40), row=1, col=1)
fig.add_trace(go.Scatter(y=density_sorted, mode="lines"), row=1, col=2)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_xaxes(title="log10(density)", row=1, col=1)
fig.update_xaxes(title="feature rank", row=1, col=2)
fig.update_layout(height=380, width=1100, title="Feature activation density",
                  showlegend=False)
fig.show()

print(f"Most-active feature density: {density.max():.4f}")
print(f"Median density:              {np.median(density):.4f}")
print(f"Fraction of dead features:   {(density < 1e-5).mean():.3f}")


## 6. Reconstruction quality vs. ground-truth recovery

The SAE successfully reconstructs activations if $\hat{x} \approx x$. But the deeper question — *do learned features correspond to the planted ground-truth features?* — is measured by the **maximum cosine similarity** between each ground-truth dictionary column and any SAE decoder column.

In [ ]:
with torch.no_grad():
    recon, codes = sae(activations)
mse = float(((activations - recon) ** 2).mean())
explained_var = 1 - float(((activations - recon) ** 2).sum() / ((activations - activations.mean(0)) ** 2).sum())
print(f"Activation reconstruction MSE: {mse:.5f}")
print(f"Explained variance:            {explained_var:.4f}")

# Cosine similarity between true dictionary atoms and SAE decoder atoms
# True dictionary is the implicit one from synth_polysemantic_activations.
# We can recover it by linear regression: D ≈ activations.T @ pinv(true_codes.T)
D_est = torch.linalg.lstsq(true_codes, activations).solution  # (F_true, d_model)
D_est = D_est / D_est.norm(dim=1, keepdim=True).clamp_min(1e-9)
D_sae = sae.W_dec.detach().T / sae.W_dec.detach().T.norm(dim=1, keepdim=True).clamp_min(1e-9)
match_matrix = torch.abs(D_est @ D_sae.T)  # (F_true, F_sae)
best_match_per_true = match_matrix.max(dim=1).values.numpy()

print(f"Mean best-match cosine (true→SAE): {best_match_per_true.mean():.4f}")
print(f"Fraction of true features with match > 0.7: {(best_match_per_true > 0.7).mean():.3f}")


## 7. Feature co-occurrence graph

Features that co-activate on the same samples are likely part of the same conceptual circuit. We restrict to the top 60 most-active SAE features and prune edges below Jaccard = 0.15.

In [ ]:
G = build_feature_cooccurrence_graph(
    codes,
    activation_threshold=0.0,
    edge_threshold=0.15,
    keep_top_features=60,
)
print(f"Graph: |V|={G.number_of_nodes()}, |E|={G.number_of_edges()}")

pos = nx.spring_layout(G, seed=42, k=0.5)
edge_x = []
edge_y = []
for u, v, _ in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_size = [10 + 60 * G.nodes[n]["density"] for n in G.nodes()]
node_text = [f"feat {n}<br>density {G.nodes[n]['density']:.4f}" for n in G.nodes()]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                          line=dict(color="rgba(120,120,120,0.4)", width=0.7),
                          hoverinfo="none"))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode="markers",
                          marker=dict(size=node_size, color=node_size,
                                      colorscale="Viridis", showscale=True,
                                      colorbar=dict(title="activation density")),
                          text=node_text, hoverinfo="text"))
fig.update_layout(height=560, width=900, showlegend=False,
                  title="SAE feature co-occurrence (force-directed)",
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()


## 8. Top-activating examples per feature

For each of the most-active features, we look up which input samples fire it most strongly. In a real LLM these would be tokens / phrases; with synthetic data we just expose the indices and their code values.

In [ ]:
density_t = result.feature_activation_density
top_features = torch.topk(density_t, 6).indices.tolist()

examples = top_activating_examples(codes, n_examples=8)
for feat_id in top_features:
    idx = examples[feat_id]
    vals = codes[idx, feat_id].tolist()
    print(f"Feature {feat_id:3d}  density={density_t[feat_id].item():.4f}")
    print(f"   top-8 sample idx={idx.tolist()}")
    print(f"   top-8 activations={[round(v, 3) for v in vals]}")


## 9. Take-aways

1. **SAE is an inverse-superposition tool.** With expansion factor $m \geq 4$ and a properly normalized decoder, an overcomplete linear autoencoder can recover planted feature codes — and we have a precise yardstick (max cosine match) to measure success.
2. **Dead features are a real failure mode.** Without periodic resampling, 20–40% of features can permanently never fire — wasted capacity. The resampling step (re-initializing to a renormalized activation sample) recovers them.
3. **Co-occurrence reveals circuits.** Once features are learned, the Jaccard graph is the natural object to navigate; force-directed layout brings circuit clusters into focus.
4. **Reconstruction MSE alone is misleading.** A perfect reconstruction can be achieved by a dense, non-monosemantic SAE. Always report sparsity + dead-feature count + ground-truth match.

### References

* Elhage, N. et al. (2022). *Toy Models of Superposition.* Anthropic.
* Bricken, T. et al. (2023). *Towards Monosemanticity.* Anthropic.
* Cunningham, H. et al. (2023). *Sparse Autoencoders Find Highly Interpretable Features in Language Models.* arXiv.
* Sundararajan, M., Taly, A., Yan, Q. (2017). *Axiomatic Attribution for Deep Networks.* ICML.
